# Text Preprocessing (Versi Lengkap & Mendalam)

Notebook ini membersihkan teks 200 artikel berita secara bertahap dan **hati-hati**. Setiap teknik pembersihan diuji dulu pada 1 contoh artikel, diterapkan ke seluruh dataset, lalu **dampaknya diukur secara eksplisit** - bukan hanya dijalankan lalu dipercaya begitu saja. Notebook ini disusun agar bisa menjadi referensi belajar Pencarian dan Penambangan Web (PPW), termasuk untuk adik tingkat yang akan mengerjakan tugas serupa.

**Alur lengkap:**

```
0. Health Check (validasi data mentah)
1. Normalisasi karakter non-ASCII (aksen, emoji)
2. Deteksi bahasa per kalimat (sebelum tanda baca dihapus)
3. Preservasi artefak penting (mata uang, kata komposit angka, angka Romawi, URL inline)
4. Case folding & penghapusan simbol
5. Sensus token pendek (1-2 huruf) - memastikan tidak ada token sampah yang lolos
6. Proteksi nama diri (termasuk partikel pada nama multi-kata)
7. Normalisasi kata tidak baku + demonstrasi pentingnya proteksi nama diri
8. Deteksi & terjemahkan kata asing
9. Statistik korpus akhir & simpan dataset
```

In [4]:
import pandas as pd
import numpy as np
import re
import time
import unicodedata
import string
import requests
from collections import Counter

df = pd.read_excel("../data/dataset_berita_200.xlsx")
print("Dataset dimuat:", df.shape)
df.head(2)

Dataset dimuat: (200, 4)


,id,isi_berita,label,url
0,1,Chef de Mission (CdM) Indonesia Todotua Pasari...,sport,https://sport.detik.com/sport-lain/d-8655308/c...
1,2,"Banjir besar melanda Nagoya, Jepang, menjelang...",sport,https://sport.detik.com/sport-lain/d-8655303/a...


In [5]:
def konteks(daftar_teks, token, window=40):
    """Mencari kemunculan pertama sebuah token di kumpulan teks, beserta potongan
    kalimat di sekitarnya, agar setiap keputusan pembersihan bisa diperiksa
    dalam konteks aslinya (bukan sekadar dipercaya begitu saja)."""
    pola = re.compile(r"(?<![A-Za-z])" + re.escape(token) + r"(?![A-Za-z])", re.IGNORECASE)
    for teks in daftar_teks:
        m = pola.search(teks)
        if m:
            mulai = max(0, m.start() - window)
            return teks[mulai:m.end() + window].replace("\n", " ")
    return "(tidak ditemukan)"

## 0. Health Check

Sebelum membersihkan teks, kita pastikan dulu data mentahnya "sehat": tidak ada nilai kosong, tidak ada baris/URL duplikat, dan `id` lengkap berurutan 1-200.

In [6]:
checks = pd.DataFrame({
    "pemeriksaan": ["jumlah baris", "nilai kosong", "isi_berita ganda", "URL ganda", "urutan id", "jumlah per label"],
    "hasil": [
        len(df), df.isna().sum().sum(), df["isi_berita"].duplicated().sum(),
        df["url"].duplicated().sum(),
        "lengkap 1-200" if list(df["id"]) == list(range(1, len(df) + 1)) else "tidak urut",
        "; ".join(f"{k}: {v}" for k, v in df["label"].value_counts().items()),
    ]
})
checks

,pemeriksaan,hasil
0,jumlah baris,200
1,nilai kosong,0
2,isi_berita ganda,0
3,URL ganda,0
4,urutan id,lengkap 1-200
5,jumlah per label,sport: 100; finance: 100


### Eksplorasi Awal: Panjang Teks Sebelum Dibersihkan

Sebelum melakukan pembersihan apa pun, kita amati dulu panjang teks per label - untuk mendeteksi anomali lebih awal (artikel nyaris kosong, ekor distribusi yang sangat panjang, dsb) sebelum anomali tersebut bercampur dengan efek pembersihan.

In [7]:
import matplotlib.pyplot as plt
import seaborn as sns

stats_awal = (df.assign(panjang_karakter=df["isi_berita"].str.len(),
                         jumlah_kata=df["isi_berita"].str.split().str.len())
                .groupby("label")[["panjang_karakter", "jumlah_kata"]]
                .agg(["min", "median", "max"]))
stats_awal

panjang_karakter               jumlah_kata             
                     min  median   max         min median   max
label                                                          
finance              826  2291.0  8169         126  315.0  1037
sport                996  2131.5  5554         140  297.0   690

## 1. Normalisasi Karakter Non-ASCII

Beberapa artikel mungkin memuat karakter di luar alfabet standar - misalnya huruf beraksen pada nama asing (`é`, `ñ`) atau emoji yang tersisip dalam kutipan (misalnya kutipan dari media sosial). Karakter-karakter ini dinormalisasi lebih dulu, sebelum tahap pembersihan simbol utama, agar tidak menimbulkan artefak aneh di teks akhir.

In [8]:
POLA_EMOJI = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U0001F900-\U0001F9FF"
    "\U00002600-\U000026FF"
    "]+", flags=re.UNICODE
)

def cek_non_ascii(teks):
    return any(ord(c) > 127 for c in teks)

def normalisasi_non_ascii(teks):
    teks = POLA_EMOJI.sub(" ", teks)
    teks = unicodedata.normalize("NFKD", teks).encode("ascii", "ignore").decode("utf-8")
    return teks

jumlah_artikel_non_ascii = df["isi_berita"].apply(cek_non_ascii).sum()
print(f"Jumlah artikel yang memuat karakter non-ASCII: {jumlah_artikel_non_ascii} dari {len(df)}")

Jumlah artikel yang memuat karakter non-ASCII: 10 dari 200


In [9]:
artikel_non_ascii = df[df["isi_berita"].apply(cek_non_ascii)]

if len(artikel_non_ascii) > 0:
    contoh_asli = artikel_non_ascii["isi_berita"].iloc[0]
else:
    contoh_asli = df["isi_berita"].iloc[0]

contoh_hasil = normalisasi_non_ascii(contoh_asli)
print("=== SEBELUM ===")
print(contoh_asli[:300])
print("\n=== SESUDAH ===")
print(contoh_hasil[:300])

=== SEBELUM ===
Memperingati Hari Ulang Tahun (HUT) ke-81 TNI tahun 2026, Komando Operasi Udara Nasional (Koopsudnas) resmi mengagendakan serangkaian kejuaraan olahraga tingkat nasional.
TNI Angkatan Udara (AU) menghadirkan berbagai kompetisi bergengsi, mulai dari Modern Pentathlon Open 2026 hingga Panahan Open 202

=== SESUDAH ===
Memperingati Hari Ulang Tahun (HUT) ke-81 TNI tahun 2026, Komando Operasi Udara Nasional (Koopsudnas) resmi mengagendakan serangkaian kejuaraan olahraga tingkat nasional.
TNI Angkatan Udara (AU) menghadirkan berbagai kompetisi bergengsi, mulai dari Modern Pentathlon Open 2026 hingga Panahan Open 202


In [10]:
df["teks_tahap1"] = df["isi_berita"].apply(normalisasi_non_ascii)
print("Normalisasi non-ASCII diterapkan ke seluruh 200 artikel.")

Normalisasi non-ASCII diterapkan ke seluruh 200 artikel.


## 2. Deteksi Bahasa per Kalimat

Sebelum tanda baca dihapus, tiap artikel dipecah menjadi kalimat, lalu bahasa dideteksi **per kalimat** - jauh lebih akurat dibanding per kata (kata tunggal seperti "mission" mudah salah dideteksi, sedangkan kalimat penuh punya cukup konteks).

In [11]:
from langdetect import detect, LangDetectException

def pecah_kalimat(teks):
    return [k.strip() for k in re.split(r"(?<=[.!?])\s+", teks) if len(k.strip()) > 0]

def deteksi_bahasa_kalimat(kalimat):
    try:
        if len(kalimat.split()) < 3:
            return "terlalu_pendek"
        return detect(kalimat)
    except LangDetectException:
        return "tidak_dikenali"

In [12]:
semua_kalimat_dengan_bahasa = []
for teks in df["teks_tahap1"]:
    for kalimat in pecah_kalimat(teks):
        semua_kalimat_dengan_bahasa.append((kalimat, deteksi_bahasa_kalimat(kalimat)))

kalimat_df = pd.DataFrame(semua_kalimat_dengan_bahasa, columns=["kalimat", "bahasa"])
print("Total kalimat di seluruh dataset:", len(kalimat_df))
print("\nDistribusi bahasa (persentase):")
print((kalimat_df["bahasa"].value_counts(normalize=True) * 100).round(1))

Total kalimat di seluruh dataset: 4171

Distribusi bahasa (persentase):
bahasa
id                95.4
terlalu_pendek     3.5
en                 0.3
tl                 0.1
pt                 0.1
de                 0.1
ro                 0.1
ca                 0.1
nl                 0.0
it                 0.0
sw                 0.0
et                 0.0
tr                 0.0
sk                 0.0
hr                 0.0
fr                 0.0
so                 0.0
da                 0.0
af                 0.0
Name: proportion, dtype: float64


**Catatan:** Bahasa `ms` (Melayu) yang muncul dalam jumlah kecil adalah hal wajar - Bahasa Indonesia dan Melayu sangat mirip sehingga algoritma deteksi bahasa kadang tertukar. Yang perlu diperhatikan adalah kalimat yang genuinely berbahasa Inggris penuh.

In [13]:
kandidat_kalimat_inggris = kalimat_df[
    (kalimat_df["bahasa"] == "en") & (kalimat_df["kalimat"].str.split().str.len() >= 5)
]
print(f"Jumlah kalimat yang diduga kuat berbahasa Inggris asli: {len(kandidat_kalimat_inggris)}")
if len(kandidat_kalimat_inggris) > 0:
    print(kandidat_kalimat_inggris["kalimat"].head(10).to_string(index=False))

Jumlah kalimat yang diduga kuat berbahasa Inggris asli: 10
               Good luck, Coach Nova," tulis PBSI.
TITF World Tennis Tour M-15 Amman Mineral Men's...
| Pos | Pebalap | Tim | Lap | \n| 1 | Kimi Anto...
Jadwal MotoGP San Marino 2026\nJumat, 11 Septem...
Mata Derrick Michael Xzavierro berbinar saat be...
Petenis Indonesia Nathan Barki melaju ke peremp...
Nice work SabRez!#BadmintonIndonesia #ChinaMast...
Berikut harga tiket untuk menonton MotoGP Indon...
               Bold Riders, Spirit of Brotherhood!
Pada 4-6 September, World Encounter Summit Bali...


## 3. Preservasi Artefak Penting

Sebelum simbol dan angka dihapus secara massal, beberapa jenis token justru **mengandung informasi penting**:

- **Mata uang** (`Rp5.000`, `US$100`) - relevan untuk artikel finance
- **Kata komposit berangka** (`3x3`, `U-18`) - relevan untuk artikel sport
- **Angka Romawi** (`III`, `IV`) - sering muncul di nama turnamen/edisi acara
- **URL inline** yang tersisip di badan teks

Token-token ini dilindungi sementara dengan kode berbasis huruf (agar selamat dari proses pembersihan simbol), lalu dikembalikan dalam bentuk ringkas setelah simbol lain dibuang.

In [14]:
def angka_ke_kode_huruf(n):
    huruf = ""
    n += 1
    while n > 0:
        n -= 1
        huruf = string.ascii_lowercase[n % 26] + huruf
        n //= 26
    return huruf

POLA_ARTEFAK = {
    "mata_uang": r"\bRp\.?\s?[\d.,]+(?:\s?(?:juta|miliar|triliun|ribu))?\b|\bUS\$\s?[\d.,]+\b",
    "komposit_angka": r"\b(?=[A-Za-z0-9-]*[A-Za-z])(?=[A-Za-z0-9-]*[0-9])[A-Za-z0-9-]{2,}\b",
    "angka_romawi": r"\b(?=[MDCLXVI])M{0,4}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})\b",
    "url_inline": r"https?://\S+|www\.\S+",
}

def lindungi_artefak(teks):
    ditemukan = {}
    urutan = 0

    def buat_pengganti(kategori):
        def pengganti(m):
            nonlocal urutan
            asli = m.group(0)
            nilai_akhir = "urltautan" if kategori == "url_inline" else re.sub(r"[^a-zA-Z0-9]", "", asli).lower()
            kode = "protectedtoken" + angka_ke_kode_huruf(urutan)
            urutan += 1
            ditemukan[kode] = nilai_akhir
            return f" {kode} "
        return pengganti

    for kategori, pola in POLA_ARTEFAK.items():
        teks = re.sub(pola, buat_pengganti(kategori), teks)
    return teks, ditemukan

def pulihkan_artefak(teks, ditemukan):
    for kode, nilai in ditemukan.items():
        teks = teks.replace(kode, nilai)
    return teks

### Hitung Total & Lihat Contoh Konteks Tiap Kategori Artefak

In [15]:
total_per_kategori = {k: 0 for k in POLA_ARTEFAK}
contoh_per_kategori = {}

for kategori, pola in POLA_ARTEFAK.items():
    for teks in df["teks_tahap1"]:
        total_per_kategori[kategori] += len(re.findall(pola, teks))
    m = re.search(pola, " ".join(df["teks_tahap1"].head(50)))
    contoh_per_kategori[kategori] = m.group(0) if m else "-"

ringkasan_artefak = pd.DataFrame({
    "kategori": list(total_per_kategori.keys()),
    "jumlah": list(total_per_kategori.values()),
    "contoh": [contoh_per_kategori[k] for k in total_per_kategori],
})
ringkasan_artefak

,kategori,jumlah,contoh
0,mata_uang,287,"Rp 103,5 juta"
1,komposit_angka,300,RS-GP26
2,angka_romawi,4374,
3,url_inline,0,-


In [16]:
daftar_teks_terlindungi, daftar_artefak = [], []
for teks in df["teks_tahap1"]:
    t, a = lindungi_artefak(teks)
    daftar_teks_terlindungi.append(t)
    daftar_artefak.append(a)

df["teks_terlindungi"] = daftar_teks_terlindungi
df["_artefak"] = daftar_artefak
print("Preservasi artefak diterapkan ke seluruh 200 artikel.")

Preservasi artefak diterapkan ke seluruh 200 artikel.


## 4. Case Folding & Penghapusan Simbol

Semua teks diubah menjadi huruf kecil dan tanda baca/simbol yang tersisa dihapus. Karena artefak penting sudah diamankan pada tahap sebelumnya, mereka akan selamat melewati proses ini.

In [17]:
def case_folding_dan_bersihkan(teks):
    teks = teks.lower()
    teks = re.sub(r"[^a-z\s]", " ", teks)
    teks = re.sub(r"\s+", " ", teks).strip()
    return teks

teks_terlindungi_contoh, artefak_ditemukan_contoh = lindungi_artefak(contoh_hasil)
contoh_bersih = case_folding_dan_bersihkan(teks_terlindungi_contoh)
contoh_pulih = pulihkan_artefak(contoh_bersih, artefak_ditemukan_contoh)

print("=== SEBELUM (artefak berupa kode) ===")
print(contoh_bersih[:250])
print("\n=== SESUDAH ARTEFAK DIPULIHKAN ===")
print(contoh_pulih[:250])

=== SEBELUM (artefak berupa kode) ===
protectedtokenc memperingati hari ulang tahun hut protectedtokena tni tahun komando operasi udara nasional koopsudnas resmi mengagendakan serangkaian kejuaraan olahraga tingkat nasional tni angkatan udara au menghadirkan berbagai kompetisi bergengsi 

=== SESUDAH ARTEFAK DIPULIHKAN ===
 memperingati hari ulang tahun hut ke81 tni tahun komando operasi udara nasional koopsudnas resmi mengagendakan serangkaian kejuaraan olahraga tingkat nasional tni angkatan udara au menghadirkan berbagai kompetisi bergengsi mulai dari  modern pentath


In [18]:
df["teks_tahap4"] = [
    pulihkan_artefak(case_folding_dan_bersihkan(t), a)
    for t, a in zip(df["teks_terlindungi"], df["_artefak"])
]
print("Case folding + pembersihan simbol diterapkan ke seluruh 200 artikel.")

Case folding + pembersihan simbol diterapkan ke seluruh 200 artikel.


## 5. Sensus Token Pendek (1-2 Huruf)

Setelah pembersihan, kita periksa token yang sangat pendek (1-2 huruf) - apakah ada "pecahan kata" yang tidak masuk akal sebagai fitur (misalnya sisa dari kata yang salah terpotong)? Token satu huruf ditetapkan tidak pernah dijadikan fitur (panjang minimum kata adalah 2 huruf) karena hampir selalu berupa pecahan/singkatan yang tidak informatif untuk klasifikasi.

In [19]:
semua_token_naif = " ".join(df["teks_tahap4"]).split()
hitung_naif = Counter(semua_token_naif)

token_1_huruf = {w: n for w, n in hitung_naif.items() if len(w) == 1}
token_2_huruf = {w: n for w, n in hitung_naif.items() if len(w) == 2}

print(f"Token total (sebelum filter panjang) : {len(semua_token_naif):,}")
print(f"Token satu huruf (jenis unik)         : {len(token_1_huruf)}  (total kemunculan: {sum(token_1_huruf.values())})")
print(f"Token dua huruf (jenis unik)          : {len(token_2_huruf)}  (total kemunculan: {sum(token_2_huruf.values())})")
print("\nContoh token dua huruf paling sering muncul (beserta konteks):")
for w, n in sorted(token_2_huruf.items(), key=lambda x: -x[1])[:8]:
    print(f"  '{w}' ({n}x) -> ...{konteks(list(df['teks_tahap4']), w)}...")

Token total (sebelum filter panjang) : 66,121
Token satu huruf (jenis unik)         : 26  (total kemunculan: 449)
Token dua huruf (jenis unik)          : 112  (total kemunculan: 2305)

Contoh token dua huruf paling sering muncul (beserta konteks):
  'di' (1298x) -> ...tua pasaribu terus memonitoring kondisi di nagoya jepang jelang asian games  itu s...
  'ke' (270x) -> ... pada september air hujan juga merembes ke beberapa venue pertandingan sementara p...
  'ia' (123x) -> ... pelayanan umum sekolah juga kata todo  ia juga menegaskan pihaknya akan terus mem...
  'pt' (67x) -> ... nanti hal senada disampaikan  direktur pt tim  indonesia emas richard sam bera  i...
  'as' (52x) -> ...rena problem teknis terdeteksi ternyata as roda mobil bmw m4 gt3 yang dikendarainy...
  'ya' (46x) -> ...m depan dan meraih gelar juara nasional ya itu harapan saya semoga tak ada aral me...
  'ri' (45x) -> ...ta ubah pola pikir kita arahan presiden ri prabowo subianto olahraga adalah duta b...
  'of' (28x) 

In [20]:
def buang_token_terlalu_pendek(teks, panjang_minimum=2):
    return " ".join(w for w in teks.split() if len(w) >= panjang_minimum)

df["teks_tahap5"] = df["teks_tahap4"].apply(buang_token_terlalu_pendek)

token_setelah = " ".join(df["teks_tahap5"]).split()
print(f"Token total setelah filter panjang minimum 2 huruf: {len(token_setelah):,} (sebelumnya {len(semua_token_naif):,})")
print("Token satu huruf yang berhasil dibuang:", sum(1 for w in semua_token_naif if len(w) == 1))

Token total setelah filter panjang minimum 2 huruf: 65,672 (sebelumnya 66,121)
Token satu huruf yang berhasil dibuang: 449


## 6. Proteksi Nama Diri

Sebelum menormalisasi kata tidak baku, kita perlu tahu dulu kata mana yang sebenarnya **nama diri**, agar tidak ikut "dibetulkan" secara keliru. Kita gunakan **bukti kapitalisasi**: kata yang muncul kapital di **tengah kalimat** (bukan di awal, karena awal kalimat selalu kapital) kemungkinan besar nama diri. Kita juga mendeteksi pola nama multi-kata dengan partikel di tengahnya (misalnya "de" pada "Chef de Mission").

In [21]:
PARTIKEL_NAMA = {"de", "van", "bin", "binti", "al", "da", "dos", "du", "el"}

def temukan_nama_diri(daftar_kalimat):
    kandidat = set()
    for kalimat in daftar_kalimat:
        kata_kalimat = kalimat.split()
        n = len(kata_kalimat)
        for i in range(n):
            kata_bersih = re.sub(r"[^A-Za-z]", "", kata_kalimat[i])
            if not kata_bersih:
                continue
            if i > 0 and kata_bersih[0].isupper() and len(kata_bersih) > 1:
                kandidat.add(kata_bersih.lower())
            if i + 2 < n:
                k1 = re.sub(r"[^A-Za-z]", "", kata_kalimat[i])
                k2 = re.sub(r"[^A-Za-z]", "", kata_kalimat[i + 1]).lower()
                k3 = re.sub(r"[^A-Za-z]", "", kata_kalimat[i + 2])
                if k1 and k2 and k3 and k1[0].isupper() and k2 in PARTIKEL_NAMA and k3[0].isupper():
                    kandidat.update([k1.lower(), k2, k3.lower()])
    return kandidat

nama_diri_terlindungi = set()
for teks in df["teks_tahap1"]:
    nama_diri_terlindungi.update(temukan_nama_diri(pecah_kalimat(teks)))

print("Total nama diri unik yang terdeteksi & dilindungi:", len(nama_diri_terlindungi))
print("Contoh:", sorted(list(nama_diri_terlindungi))[:20])
print("\nPartikel nama yang berhasil ikut terlindungi (dari nama multi-kata):",
      sorted(nama_diri_terlindungi & PARTIKEL_NAMA))

Total nama diri unik yang terdeteksi & dilindungi: 3104
Contoh: ['aadi', 'aam', 'aan', 'aau', 'abadi', 'abang', 'abangpalmerah', 'abdi', 'abdul', 'aberdeen', 'abraham', 'abu', 'abullah', 'accurate', 'aceh', 'acehsumut', 'achmad', 'acid', 'acosta', 'activ']

Partikel nama yang berhasil ikut terlindungi (dari nama multi-kata): ['al', 'bin', 'da', 'de', 'el', 'van']


## 7. Normalisasi Kata Tidak Baku

Sekarang kita normalisasi kata-kata informal/tidak baku ke bentuk baku, menggunakan kamus normalisasi kata Bahasa Indonesia (open-source). Kata yang termasuk **nama diri terlindungi** atau **kemungkinan singkatan resmi** dikecualikan.

In [22]:
url_kamus = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
resp = requests.get(url_kamus)
with open("kamus_alay.csv", "wb") as f:
    f.write(resp.content)

kamus_df = pd.read_csv("kamus_alay.csv")
kamus_normalisasi = dict(zip(kamus_df["slang"], kamus_df["formal"]))
print("Jumlah entri kamus normalisasi:", len(kamus_df))

Jumlah entri kamus normalisasi: 15006


### Demonstrasi: Kenapa Proteksi Nama Diri Penting?

Sebelum menerapkan normalisasi secara aman, mari kita lihat dulu apa yang terjadi **tanpa** proteksi nama diri - untuk membuktikan bahwa langkah proteksi ini benar-benar diperlukan, bukan sekadar langkah tambahan tanpa manfaat nyata.

In [23]:
def normalisasi_naif(teks):
    return " ".join(kamus_normalisasi.get(k, k) for k in teks.split())

contoh_teks5 = df["teks_tahap5"].iloc[0]
hasil_naif = normalisasi_naif(contoh_teks5)
kata_asli = contoh_teks5.split()
kata_naif = hasil_naif.split()

perubahan_naif = [(a, b) for a, b in zip(kata_asli, kata_naif) if a != b]
print(f"Tanpa proteksi nama diri, {len(perubahan_naif)} kata berubah pada contoh artikel ini:")
for asli, baru in perubahan_naif[:10]:
    print(f"  '{asli}' -> '{baru}'")

Tanpa proteksi nama diri, 1 kata berubah pada contoh artikel ini:
  'de' -> 'deh'


In [24]:
kata_dikecualikan_normalisasi = set()

def kemungkinan_singkatan(kata, teks_asli_gabungan):
    if len(kata) > 5:
        return False
    pola_kapital_penuh = re.findall(rf"\b{kata.upper()}\b", teks_asli_gabungan)
    pola_kecil = re.findall(rf"\b{kata}\b", teks_asli_gabungan.lower())
    if len(pola_kecil) == 0:
        return False
    return (len(pola_kapital_penuh) / len(pola_kecil)) > 0.5

teks_asli_gabungan = " ".join(df["isi_berita"])

def normalisasi_kata_aman(teks):
    hasil = []
    for kata in teks.split():
        if kata in nama_diri_terlindungi or kemungkinan_singkatan(kata, teks_asli_gabungan):
            kata_dikecualikan_normalisasi.add(kata)
            hasil.append(kata)
        else:
            hasil.append(kamus_normalisasi.get(kata, kata))
    return " ".join(hasil)

hasil_aman = normalisasi_kata_aman(contoh_teks5)
kata_aman = hasil_aman.split()
perubahan_aman = [(a, b) for a, b in zip(kata_asli, kata_aman) if a != b]

print(f"Dengan proteksi nama diri, hanya {len(perubahan_aman)} kata yang benar-benar berubah:")
for asli, baru in perubahan_aman[:10]:
    print(f"  '{asli}' -> '{baru}'")

kata_terselamatkan = set(a for a, _ in perubahan_naif) - set(a for a, _ in perubahan_aman)
print(f"\nKata yang berhasil diselamatkan dari normalisasi keliru: {sorted(kata_terselamatkan)}")

Dengan proteksi nama diri, hanya 0 kata yang benar-benar berubah:

Kata yang berhasil diselamatkan dari normalisasi keliru: ['de']


In [25]:
df["teks_tahap7"] = df["teks_tahap5"].apply(normalisasi_kata_aman)

total_kata_semua = len(" ".join(df["teks_tahap5"]).split())
total_kata_berubah = sum(1 for a, b in zip(
    " ".join(df["teks_tahap5"]).split(), " ".join(df["teks_tahap7"]).split()
) if a != b)

print("Normalisasi diterapkan ke seluruh 200 artikel.")
print(f"Kata yang benar-benar dibakukan : {total_kata_berubah} dari {total_kata_semua:,} total kata")
print(f"Kata dikecualikan (nama diri/singkatan): {len(kata_dikecualikan_normalisasi)}")

Normalisasi diterapkan ke seluruh 200 artikel.
Kata yang benar-benar dibakukan : 84 dari 65,672 total kata
Kata dikecualikan (nama diri/singkatan): 2995


## 8. Deteksi & Terjemahkan Kata Asing

Kata-kata unik yang terdeteksi Bahasa Inggris (dan bukan nama diri) dikumpulkan sekali, lalu diterjemahkan sekali per kata menggunakan WordNet sebagai sumber utama dan Google Translate sebagai cadangan.

In [26]:
import nltk
nltk.download("words", quiet=True)
from nltk.corpus import words as nltk_words

kamus_inggris = set(w.lower() for w in nltk_words.words())

url_kbbi = "https://raw.githubusercontent.com/sastrawi/sastrawi/master/data/kata-dasar.txt"
resp_kbbi = requests.get(url_kbbi)
kamus_indonesia = set(resp_kbbi.text.strip().split("\n"))

def kemungkinan_bahasa_asing(kata):
    if len(kata) <= 3 or kata in nama_diri_terlindungi:
        return False
    return (kata not in kamus_indonesia) and (kata in kamus_inggris)

semua_kata = set()
for teks in df["teks_tahap7"]:
    semua_kata.update(teks.split())

kata_asing = [k for k in semua_kata if kemungkinan_bahasa_asing(k)]
print(f"Total kata unik di dataset: {len(semua_kata)}")
print(f"Kata terdeteksi bahasa asing (nama diri sudah dikecualikan): {len(kata_asing)}")
print("Contoh:", kata_asing[:20])

Total kata unik di dataset: 8213
Kata terdeteksi bahasa asing (nama diri sudah dikecualikan): 240
Contoh: ['host', 'inflow', 'happy', 'year', 'ammonia', 'agent', 'acuan', 'break', 'basically', 'booth', 'looping', 'progress', 'handling', 'confidence', 'endorsement', 'driver', 'backhand', 'meng', 'beneficiary', 'stream']


In [27]:
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
try:
    nltk.download("omw-2.0", quiet=True)
except Exception:
    pass
from nltk.corpus import wordnet as wn
from deep_translator import GoogleTranslator

translator = GoogleTranslator(source="en", target="id")

def terjemahkan_wordnet(kata):
    for syn in wn.synsets(kata, lang="eng"):
        lemma_id = syn.lemma_names("ind")
        if lemma_id:
            return lemma_id[0].replace("_", " ")
    return None

kamus_terjemahan = {}
sumber_terjemahan = {"wordnet": 0, "google_translate": 0, "gagal": 0}

for i, kata in enumerate(kata_asing):
    hasil = terjemahkan_wordnet(kata)
    if hasil is not None:
        sumber_terjemahan["wordnet"] += 1
    else:
        try:
            time.sleep(0.3)
            hasil = translator.translate(kata).lower()
            if hasil != kata:
                sumber_terjemahan["google_translate"] += 1
            else:
                sumber_terjemahan["gagal"] += 1
        except Exception:
            hasil = kata
            sumber_terjemahan["gagal"] += 1
    kamus_terjemahan[kata] = hasil
    if (i + 1) % 50 == 0:
        print(f"Progress: {i+1}/{len(kata_asing)}")

print("Selesai membangun kamus terjemahan.")
print("Sumber terjemahan:", sumber_terjemahan)

Progress: 50/240
Progress: 100/240
Progress: 150/240
Progress: 200/240
Selesai membangun kamus terjemahan.
Sumber terjemahan: {'wordnet': 214, 'google_translate': 0, 'gagal': 26}


In [28]:
def terjemahkan_teks(teks):
    return " ".join(kamus_terjemahan.get(k, k) for k in teks.split())

df["teks_bersih"] = df["teks_tahap7"].apply(terjemahkan_teks)

print("=== CONTOH HASIL AKHIR (SEMUA TAHAP) ===")
print(df["teks_bersih"].iloc[0][:400])

=== CONTOH HASIL AKHIR (SEMUA TAHAP) ===
chef de mission cdm indonesia todotua pasaribu terus memonitoring kondisi di nagoya jepang jelang asian games itu setelah adanya banjir besar yang menimpa kota tersebut hal tersebut ditekankan todo sapaan karibnya karena berkaitan dengan kontingen indonesia yang akan tampil di asian games mengutip laman straitstimes ratusan atlet asian games di jepang sempat dievakuasi dari tempat penginapan merek


## 9. Statistik Korpus Akhir & Simpan Dataset

In [29]:
semua_token_final = " ".join(df["teks_bersih"]).split()
total_token_final = len(semua_token_final)
total_kata_unik_final = len(set(semua_token_final))

print(f"Total token di seluruh korpus : {total_token_final:,}")
print(f"Total kata unik (vocabulary)  : {total_kata_unik_final:,}")
print(f"Type-Token Ratio (TTR)        : {total_kata_unik_final/total_token_final:.4f}")

Total token di seluruh korpus : 65,708
Total kata unik (vocabulary)  : 8,121
Type-Token Ratio (TTR)        : 0.1236


In [30]:
ringkasan_akhir = pd.DataFrame({
    "metrik": ["dokumen", "token mentah (naif)", "token setelah filter panjang", "kata unik akhir",
               "nama diri terlindungi", "kata dibakukan", "kata asing diterjemahkan",
               "artefak diselamatkan (total)"],
    "nilai": [len(df), len(semua_token_naif), total_token_final, total_kata_unik_final,
              len(nama_diri_terlindungi), total_kata_berubah, len(kata_asing),
              sum(total_per_kategori.values())]
})
ringkasan_akhir

,metrik,nilai
0,dokumen,200
1,token mentah (naif),66121
2,token setelah filter panjang,65708
3,kata unik akhir,8121
4,nama diri terlindungi,3104
5,kata dibakukan,84
6,kata asing diterjemahkan,240
7,artefak diselamatkan (total),4961


In [31]:
kolom_simpan = ["id", "label", "url", "teks_bersih"]
df[kolom_simpan].to_excel("../data/dataset_berita_200_bersih.xlsx", index=False)
print("Dataset hasil preprocessing disimpan ke data/dataset_berita_200_bersih.xlsx")

Dataset hasil preprocessing disimpan ke data/dataset_berita_200_bersih.xlsx


### Ringkasan Tahapan

| Tahap | Tujuan |
|---|---|
| 0. Health Check | Memastikan data mentah bebas nilai kosong & duplikat |
| 1. Non-ASCII | Membersihkan aksen & emoji |
| 2. Deteksi bahasa per kalimat | Mengidentifikasi kalimat asing secara akurat |
| 3. Preservasi artefak | Menyelamatkan mata uang, kode kategori, angka Romawi, URL |
| 4. Case folding & simbol | Menyeragamkan huruf kecil, membuang tanda baca |
| 5. Sensus token pendek | Membuang token 1 huruf yang tidak informatif |
| 6. Proteksi nama diri | Mencegah nama orang/tempat salah dinormalisasi |
| 7. Normalisasi kata tidak baku | Membakukan kata informal (dibuktikan dengan demo before/after) |
| 8. Terjemahan kata asing | Mengubah istilah Inggris menjadi Bahasa Indonesia |
| 9. Statistik & simpan | Dataset bersih siap untuk tahap Vectorization |